In [ ]:
import json
import os
import random
import time
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup, get_constant_schedule, get_constant_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATASET = "SST-2"
MODEL = "BERT"
MODEL_NAME = "bert-base-uncased"
NUM_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 0
OPTIMIZER_NAME = "AdamW"
SCHEDULER_NAME = "linear"
TEST_LABELS = False

TEXT_COLUMN = "sentence_original"
LABEL_COLUMN = "label"
ROOT_DIR = os.getcwd()

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Dataset:       {DATASET}")
print(f"Model:         {MODEL_NAME}")
print(f"Epochs:        {NUM_EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max length:    {MAX_LENGTH}")
print(f"Weight decay:  {WEIGHT_DECAY}")
print(f"Warmup steps:  {WARMUP_STEPS}")
print(f"Optimizer:     {OPTIMIZER_NAME}")
print(f"Scheduler:     {SCHEDULER_NAME}")
print(f"Test labels:   {TEST_LABELS}")
print(f"Seed:          {SEED}")
print(f"Root dir:      {ROOT_DIR}")

In [ ]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    file_path = os.path.join(dataset_path, f"{split_name}.json")
    if not os.path.exists(file_path):
        print(f"  {split_name}.json not found in {dataset_path}")
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  {split_name}.json: {len(df)} samples")
    return df

print(f"Loading data from: {dataset_path}\n")
df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

if df_train is not None:
    print(f"\nFirst 3 train examples:")
    display(df_train.head(3))

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class SSTDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_texts = df_train[TEXT_COLUMN].fillna("").tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts = df_validation[TEXT_COLUMN].fillna("").tolist()
val_labels = df_validation[LABEL_COLUMN].tolist()
test_texts = df_test[TEXT_COLUMN].fillna("").tolist()

y_test = None
if TEST_LABELS and LABEL_COLUMN in df_test.columns:
    y_test = df_test[LABEL_COLUMN].tolist()

train_dataset = SSTDataset(train_texts, train_labels)
val_dataset = SSTDataset(val_texts, val_labels)
test_dataset = SSTDataset(test_texts, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
num_labels = len(set(train_labels))
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
model.to(device)

total_steps = len(train_loader) * NUM_EPOCHS

def build_optimizer(name, model_params, lr, weight_decay):
    optimizers = {
        "AdamW": lambda: torch.optim.AdamW(model_params, lr=lr, weight_decay=weight_decay),
        "Adam": lambda: torch.optim.Adam(model_params, lr=lr, weight_decay=weight_decay),
        "SGD": lambda: torch.optim.SGD(model_params, lr=lr, weight_decay=weight_decay, momentum=0.9),
    }
    if name not in optimizers:
        raise ValueError(f"Unknown optimizer: {name}. Choose from: {list(optimizers.keys())}")
    return optimizers[name]()

def build_scheduler(name, optimizer, warmup_steps, total_steps):
    schedulers = {
        "linear": lambda: get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps),
        "cosine": lambda: get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps),
        "constant": lambda: get_constant_schedule(optimizer),
        "constant_with_warmup": lambda: get_constant_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps),
        "none": lambda: None,
    }
    if name not in schedulers:
        raise ValueError(f"Unknown scheduler: {name}. Choose from: {list(schedulers.keys())}")
    return schedulers[name]()

optimizer = build_optimizer(OPTIMIZER_NAME, model.parameters(), LEARNING_RATE, WEIGHT_DECAY)
scheduler = build_scheduler(SCHEDULER_NAME, optimizer, WARMUP_STEPS, total_steps)

optimizer_info = {"name": OPTIMIZER_NAME, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
for k, v in optimizer.defaults.items():
    if k not in optimizer_info:
        optimizer_info[k] = v

scheduler_info = {"name": SCHEDULER_NAME, "warmup_steps": WARMUP_STEPS, "total_steps": total_steps}

print(f"Num labels:   {num_labels}")
print(f"Total steps:  {total_steps}")
print(f"Optimizer:    {OPTIMIZER_NAME}")
print(f"Scheduler:    {SCHEDULER_NAME}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print(f"\nTraining {MODEL} for {NUM_EPOCHS} epochs...\n")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0
    epoch_preds, epoch_labels = [], []

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        epoch_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        epoch_preds.extend(preds.cpu().numpy())
        epoch_labels.extend(labels.cpu().numpy())

    train_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(epoch_labels, epoch_preds)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval()
    val_loss = 0
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_labels_list, val_preds)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"  Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

train_time = time.time() - start_time
entries_per_second = (len(train_labels) * NUM_EPOCHS) / train_time

print(f"\nTraining complete in {train_time:.2f} seconds")
print(f"  Entries/sec: {entries_per_second:.0f}")
print(f"  Final Train Acc: {train_accs[-1]:.4f}")
print(f"  Final Val Acc:   {val_accs[-1]:.4f}")

In [ ]:
model.eval()

def predict(loader):
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            has_labels = "labels" in batch and batch["labels"] is not None
            if has_labels:
                labels = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                all_labels.extend(labels.cpu().numpy())
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
    avg_loss = total_loss / len(loader) if all_labels else None
    return np.array(all_preds), np.array(all_labels) if all_labels else None, avg_loss

y_train_pred, y_train_true, train_loss_final = predict(train_loader)
y_val_pred, y_val_true, val_loss_final = predict(val_loader)
y_test_pred, y_test_true, test_loss_final = predict(test_loader)

train_accuracy = accuracy_score(y_train_true, y_train_pred)

val_accuracy = accuracy_score(y_val_true, y_val_pred)
val_precision_macro = precision_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_recall_macro = recall_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_f1_macro = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
val_precision_per_class = precision_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_recall_per_class = recall_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_f1_per_class = f1_score(y_val_true, y_val_pred, average=None, zero_division=0).tolist()
val_conf_matrix = confusion_matrix(y_val_true, y_val_pred).tolist()

class_labels = sorted(np.unique(y_val_true).tolist())

if TEST_LABELS and y_test_true is not None:
    test_accuracy = accuracy_score(y_test_true, y_test_pred)
    test_precision_macro = precision_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_recall_macro = recall_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test_true, y_test_pred, average="macro", zero_division=0)
    test_precision_per_class = precision_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_recall_per_class = recall_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_f1_per_class = f1_score(y_test_true, y_test_pred, average=None, zero_division=0).tolist()
    test_conf_matrix = confusion_matrix(y_test_true, y_test_pred).tolist()

print("=" * 60)
print(f"  RESULTS {MODEL} on {DATASET}")
print("=" * 60)
print(f"  Train Accuracy:      {train_accuracy:.4f}   | Loss: {train_loss_final:.4f}")
print(f"  Val Accuracy:        {val_accuracy:.4f}   | Loss: {val_loss_final:.4f}")
if TEST_LABELS and y_test_true is not None:
    print(f"  Test Accuracy:       {test_accuracy:.4f}   | Loss: {test_loss_final:.4f}")
    print(f"  Test Precision:      {test_precision_macro:.4f}")
    print(f"  Test Recall:         {test_recall_macro:.4f}")
    print(f"  Test F1 (macro):     {test_f1_macro:.4f}")
    print(f"\n  Test Confusion Matrix: {test_conf_matrix}")
    print(f"\n  Classification Report (TEST):")
    print(classification_report(y_test_true, y_test_pred, zero_division=0))
else:
    unique, counts = np.unique(y_test_pred, return_counts=True)
    print(f"  Test prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")
    print(f"\n  Classification Report (VALIDATION):")
    print(classification_report(y_val_true, y_val_pred, zero_division=0))

In [ ]:
if TEST_LABELS and y_test_true is not None:
    cm_array = np.array(test_conf_matrix)
    cm_labels = sorted(np.unique(y_test_true).tolist())
else:
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val_true).tolist())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
ax_cm.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
fig_cm.tight_layout()
plt.show()

print(f"Classes: {cm_labels}")
print(f"Matrix:\n{cm_array}")

In [ ]:
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_acc, ax_acc = plt.subplots(figsize=(9, 5))
ax_acc.plot(epochs_range, train_accs, label="Train Accuracy", linewidth=2, color="#1f77b4")
ax_acc.plot(epochs_range, val_accs, label="Validation Accuracy", linewidth=2, color="#ff7f0e")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy")
ax_acc.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_acc.legend(loc="lower right")
ax_acc.grid(True, alpha=0.3)
fig_acc.tight_layout()
plt.show()

print(f"Last epoch -- Train: {train_accs[-1]:.4f}, Val: {val_accs[-1]:.4f}")

In [ ]:
fig_loss, ax_loss = plt.subplots(figsize=(9, 5))
ax_loss.plot(epochs_range, train_losses, label="Train Loss", linewidth=2, color="#1f77b4")
ax_loss.plot(epochs_range, val_losses, label="Validation Loss", linewidth=2, color="#ff7f0e")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss (CrossEntropy)")
ax_loss.set_title(f"{MODEL} - {DATASET}", fontweight="bold", color="black")
ax_loss.legend(loc="upper right")
ax_loss.grid(True, alpha=0.3)
fig_loss.tight_layout()
plt.show()

print(f"Last epoch -- Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

In [ ]:
def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, model_hf_name, num_epochs, batch_size, learning_rate, max_length,
                 test_labels, text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 optimizer_info=None, scheduler_info=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    model_size_mb = model_size_bytes / (1024 * 1024)

    timestamp_str = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    run_name = f"{model_hf_name}_ep{num_epochs}_bs{batch_size}_lr{learning_rate}_{timestamp_str}"

    output_dir = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(output_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(output_dir, fig_name), dpi=150, bbox_inches="tight")
    print(f"Charts saved: {list(figures.keys())}")

    total_params = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    metrics = {
        "model": model_name,
        "model_hf_name": model_hf_name,
        "dataset": dataset,
        "run_name": run_name,
        "timestamp": timestamp_str,
        "seed": SEED,
        "model_params": {
            "num_epochs": num_epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "max_length": max_length,
            "total_params": total_params,
            "trainable_params": trainable_params,
        },
        "model_size": {
            "bytes": model_size_bytes,
            "MB": round(model_size_mb, 4)
        },
        "training": {
            "training_time_seconds": round(train_time, 4),
            "entries_per_second": round(entries_per_second, 2),
            "num_epochs": num_epochs,
            "train_samples": len(df_train),
        },
        "data": {
            "train_samples": len(df_train),
            "validation_samples": len(df_validation),
            "test_samples": len(df_test) if df_test is not None else 0,
            "text_column": text_column,
            "label_column": label_column,
            "classes": class_labels,
        },
        "optimizer": optimizer_info or {},
        "scheduler": scheduler_info or {},
        "train_results": train_results,
        "validation_results": val_results,
    }

    if test_labels and test_results is not None:
        metrics["test_results"] = test_results

    metrics_path = os.path.join(output_dir, "metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"metrics.json saved in: {output_dir}")
    return output_dir, metrics_path, run_name


figures = {
    "confusion_matrix.png": fig_cm,
    "accuracy_curves.png": fig_acc,
    "loss_curves.png": fig_loss,
}

train_res = {
    "accuracy": round(train_accuracy, 4),
    "loss": round(train_loss_final, 4),
}

val_res = {
    "accuracy": round(val_accuracy, 4),
    "loss": round(val_loss_final, 4),
    "precision_macro": round(val_precision_macro, 4),
    "recall_macro": round(val_recall_macro, 4),
    "f1_macro": round(val_f1_macro, 4),
    "precision_per_class": [round(p, 4) for p in val_precision_per_class],
    "recall_per_class": [round(r, 4) for r in val_recall_per_class],
    "f1_per_class": [round(f, 4) for f in val_f1_per_class],
    "confusion_matrix": val_conf_matrix,
}

test_res = None
if TEST_LABELS and y_test_true is not None:
    test_res = {
        "accuracy": round(test_accuracy, 4),
        "loss": round(test_loss_final, 4),
        "precision_macro": round(test_precision_macro, 4),
        "recall_macro": round(test_recall_macro, 4),
        "f1_macro": round(test_f1_macro, 4),
        "precision_per_class": [round(p, 4) for p in test_precision_per_class],
        "recall_per_class": [round(r, 4) for r in test_recall_per_class],
        "f1_per_class": [round(f, 4) for f in test_f1_per_class],
        "confusion_matrix": test_conf_matrix,
    }

output_dir, metrics_path, run_name = save_results(
    model_obj=model, figures=figures,
    train_results=train_res, val_results=val_res, test_results=test_res,
    train_time=train_time, entries_per_second=entries_per_second,
    dataset=DATASET, model_name=MODEL, model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    test_labels=TEST_LABELS, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
    class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test,
    root_dir=ROOT_DIR,
    optimizer_info=optimizer_info, scheduler_info=scheduler_info,
)

In [ ]:
with open(metrics_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"{'='*60}")
print(f"  RUN SUMMARY: {run_name}")
print(f"{'='*60}")
print(f"  Model:              {saved['model']}")
print(f"  HF Model:           {saved['model_hf_name']}")
print(f"  Dataset:            {saved['dataset']}")
print(f"  Training time:      {saved['training']['training_time_seconds']}s")
print(f"  Entries/sec:        {saved['training']['entries_per_second']}")
print(f"  Model size:         {saved['model_size']['MB']} MB")
print(f"  Total params:       {saved['model_params']['total_params']:,}")
print(f"  Train Accuracy:     {saved['train_results']['accuracy']}")
print(f"  Train Loss:         {saved['train_results']['loss']}")
print(f"  Val Accuracy:       {saved['validation_results']['accuracy']}")
print(f"  Val Loss:           {saved['validation_results']['loss']}")
print(f"  Val F1 (macro):     {saved['validation_results']['f1_macro']}")
if "test_results" in saved:
    print(f"  Test Accuracy:      {saved['test_results']['accuracy']}")
    print(f"  Test Loss:          {saved['test_results']['loss']}")
    print(f"  Test F1 (macro):    {saved['test_results']['f1_macro']}")
else:
    print(f"  Test:               N/A (TEST_LABELS=False)")
print(f"{'='*60}")
print(f"\n  Files saved in: {output_dir}")
for f_name in os.listdir(output_dir):
    f_size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"    {f_name} ({f_size:,} bytes)")

In [ ]:
print(json.dumps(saved, indent=2, ensure_ascii=False))